In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install langchain-chroma

In [ ]:
!pip install -q ragas==0.4.3 langchain==0.3.30 langchain-community==0.3.31 langchain-openai==0.3.35 rapidfuzz python-dotenv sentence-transformers pandas langchain-text-splitters
print("[✓] Tüm kütüphaneler başarıyla kuruldu.")

In [ ]:
!cp /kaggle/input/notebooks/""/sonuc-analizi-retriever/chunk_map_20260829_060905.csv .
!cp /kaggle/input/notebooks/""/sonuc-analizi-retriever/testset_mixed_102_soru_20260829_060905.csv .
!cp -r /kaggle/input/notebooks/""/sonuc-analizi-retriever/chroma_db_contextual .
!cp -r /kaggle/input/notebooks/""/sonuc-analizi-retriever/chroma_db .

!cp /kaggle/input/notebooks/""/rag-ve-de-erlendirme/tahminler.csv .

In [ ]:
!mkdir outputs

In [ ]:
!pip install rank_bm25
!pip install sentence-transformers

In [ ]:
import os
import sys
import datetime
import time
import asyncio
import hashlib
import pandas as pd

from openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.messages import BaseMessage, AIMessage

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.run_config import RunConfig
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer, 
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)

import shutil
import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma

import ast
import time
import asyncio
import pandas as pd
from openai import OpenAI
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
answer_correctness
)

In [ ]:
print("="*70)
print("   UÇTAN UCA 3'LÜ SİSTEM RAGAS DEĞERLENDİRME PİPELİNE'I (DÜZELTİLDİ)")
print("="*70)

# 1. Özel Azure Responses Model Tanımlaması (gpt-4.1-mini)
from langchain_core.outputs import ChatResult, ChatGeneration
from openai import AsyncOpenAI

class CustomAzureResponsesModel(BaseChatModel):
    api_key: str = ""
    base_url: str = ""
    model_name: str = "gpt-4.1-mini"
    delay: float = 0.2
    def _generate(self, messages, stop=None, **kwargs):
        time.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])
        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        response = client.responses.create(model=self.model_name, input=prompt_text)
        output_text = getattr(response, "output_text", str(response))
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=output_text))])

    async def _agenerate(self, messages, stop=None, **kwargs):
        await asyncio.sleep(self.delay)
        prompt_text = "\n".join([msg.content for msg in messages])
        async_client = AsyncOpenAI(api_key=self.api_key, base_url=self.base_url)
        response = await async_client.responses.create(model=self.model_name, input=prompt_text)
        output_text = getattr(response, "output_text", str(response))
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=output_text))])

    @property
    def _llm_type(self) -> str:
        return "custom-azure-responses-openai"

   
llm = CustomAzureResponsesModel()

In [ ]:
# 2. Dosya Yolları ve Veri Setleri
chunk_map_path = "/kaggle/working/chunk_map_20260829_060905.csv"
testset_path = "/kaggle/working/testset_mixed_102_soru_20260829_060905.csv"
db_dir = "/kaggle/working/chroma_db"
contextual_db_dir = "/kaggle/working/chroma_db_contextual"

df_map = pd.read_csv(chunk_map_path)
df_test = pd.read_csv(testset_path)

# Test setindeki sütunları kontrol edelim (Doğru ground truth sütununu yakalamak için)
print(f"[*] Test seti sütunları: {df_test.columns.tolist()}")

In [ ]:
docs_for_bm25 = []
for _, row in df_map.iterrows():
    clean_text = str(row['text']).strip()
    chunk_id = str(row['chunk_id'])
    docs_for_bm25.append(Document(page_content=clean_text, metadata={"chunk_id": chunk_id}))

# 3. Embedding ve Veritabanı Bağlantıları
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

db = Chroma(persist_directory=db_dir, embedding_function=embeddings)
db_contextual = Chroma(persist_directory=contextual_db_dir, embedding_function=embeddings)

contextual_data = db_contextual.get()
docs_for_ctx_bm25 = []
for doc_text, metadata in zip(contextual_data['documents'], contextual_data['metadatas']):
    docs_for_ctx_bm25.append(Document(page_content=doc_text, metadata={"chunk_id": str(metadata.get('chunk_id'))}))

In [ ]:
from langchain_community.retrievers import BM25Retriever
import pandas as pd
import ast
import numpy as np
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Reranker modülleri
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers import ContextualCompressionRetriever

K = 5
INITIAL_K_FOR_RERANK = 15


basic_retriever = db.as_retriever(search_kwargs={"k": K})

# Normal Hibrit (Contextual Olmayan: Standart BM25 + Standart Vektör)
bm25_std = BM25Retriever.from_documents(docs_for_bm25)
bm25_std.k = K
vector_std = db.as_retriever(search_kwargs={"k": K})
normal_hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_std, vector_std], weights=[0.5, 0.5]
)

# Contextual + Hibrit (Saf)
bm25_ctx = BM25Retriever.from_documents(docs_for_ctx_bm25)
bm25_ctx.k = K
contextual_chroma_std = db_contextual.as_retriever(search_kwargs={"k": K})
contextual_hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_ctx, contextual_chroma_std], weights=[0.5, 0.5]
)

# Ctx + Hibrit + Reranker (Saf)
bm25_ctx_wide = BM25Retriever.from_documents(docs_for_ctx_bm25)
bm25_ctx_wide.k = INITIAL_K_FOR_RERANK
contextual_wide_dense = db_contextual.as_retriever(search_kwargs={"k": INITIAL_K_FOR_RERANK})
contextual_hybrid_wide = EnsembleRetriever(
    retrievers=[bm25_ctx_wide, contextual_wide_dense], weights=[0.5, 0.5]
)

reranker_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3", model_kwargs={'device': 'cuda'})
compressor = CrossEncoderReranker(model=reranker_model, top_n=K)
contextual_reranker_pure = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=contextual_hybrid_wide)

retrievers_to_test = {
    "1. Temel Vektör": basic_retriever,
    "2. Normal Hibrit (Contextual Yok)": normal_hybrid_retriever,
    "3. Contextual + Hibrit (Saf)": contextual_hybrid_retriever,
    "4. Ctx + Hibrit + Reranker (Saf)": contextual_reranker_pure
}

In [ ]:
'''
# 5. Test, Üretim ve Değerlendirme Döngüsü
eval_data = []
total_soru = len(df_test)
print(f"\n[*] Toplam {total_soru} soru işlenecek. İlerleme her 10 soruda bir raporlanacaktır.\n" + "-"*60)

for system_name, retriever in retrievers_to_test.items():
    print(f"\n🔄 [SİSTEM BAŞLADI] Mimari: {system_name}")
    print(f"   -> 102 soru için gpt-4.1-mini yanıtları üretiliyor, lütfen bekleyin...")
    
    count = 0
    for index, row in df_test.iterrows():
        query = row['user_input']
        
        # RAGAS ground truth (ideal cevap metni) sütununu güvenli bir şekilde alıyoruz
        ground_truth_answer = ""
        for gt_col in ['ground_truth', 'response', 'answer', 'ideal_answer']:
            if gt_col in df_test.columns and pd.notna(row[gt_col]):
                ground_truth_answer = str(row[gt_col])
                break
        
        # Eğer yukarıdakiler bulunamazsa fallback olarak reference_contexts stringleştirilir
        if not ground_truth_answer and 'reference_contexts' in df_test.columns:
            try:
                gt_list = ast.literal_eval(row['reference_contexts'])
                ground_truth_answer = " ".join(gt_list)
            except Exception:
                ground_truth_answer = str(row['reference_contexts'])

        if not ground_truth_answer:
            continue

        # A) Retrieval
        retrieved_docs = retriever.invoke(query)
        contexts = [doc.page_content for doc in retrieved_docs[:K]]
        if not contexts:
            continue

        # B) Generation (gpt-4.1-mini)
        context_text = "\n\n---\n\n".join(contexts)
        prompt_text = f"""You are a helpful AI assistant. Use the following context to answer the user's question accurately. If you don't know based on the context, say that you don't know.

Context:
{context_text}

Question: {query}

Answer:"""
        
        response_msg = llm.invoke(prompt_text)
        answer = response_msg.content

        eval_data.append({
            "system_name": system_name,
            "question": query,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": ground_truth_answer # RAGAS'ın beklediği tekil metin formatı
        })
        count += 1
        # --- İlerleme Takibi İçin Eklenen Print ---
        if count % 10 == 0:
            print(f"   ... {count}/{total_soru} soru işlendi.")
    
    print(f"   [✓] {system_name} için {count} soru başarıyla tamamlandı.")

df_eval_raw = pd.DataFrame(eval_data)
df_eval_raw.to_csv("tahminler.csv")
'''

In [ ]:
df_eval_raw = pd.read_csv("tahminler.csv")

In [ ]:
print("\n" + "="*60)
print("📊 Tüm sistemlerin yanıt üretimi bitti. RAGAS metrikleri hesaplanıyor...")
print("="*60)

import ast
from ragas import RunConfig
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    ResponseRelevancy,
    AnswerCorrectness,
)

# Zaman aşımı süresini ve worker sayısını ayarlıyoruz
config = RunConfig(timeout=300, max_retries=10, max_workers=2)

# Metrikleri parantezli şekilde (nesne olarak) başlatıyoruz:
metrics = [
    ContextPrecision(),
    ContextRecall(),
    Faithfulness(),
    ResponseRelevancy(),
    AnswerCorrectness(),
]

system_scores = []
total_sistem = len(retrievers_to_test)
curr_idx = 0

for system_name, retriever in retrievers_to_test.items():
    curr_idx += 1
    print(f"\n------------------------------------------------------------")
    print(f"⏳ [{curr_idx}/{total_sistem}] RAGAS skorları hesaplanıyor: {system_name}")
    print(f"   -> Bu işlem birkaç dakika sürebilir, lütfen bekleyin...")
    print(f"------------------------------------------------------------")
    
    sub_df = df_eval_raw[df_eval_raw["system_name"] == system_name]
    
    # Contexts verisinin string kalma ihtimaline karşı güvenli liste dönüşümü
    raw_contexts = sub_df["contexts"].tolist()
    processed_contexts = []
    for ctx in raw_contexts:
        if isinstance(ctx, str):
            try:
                processed_contexts.append(ast.literal_eval(ctx))
            except:
                processed_contexts.append([ctx])
        else:
            processed_contexts.append(ctx)

    dataset_dict = {
        "question": sub_df["question"].tolist(),
        "answer": sub_df["answer"].tolist(),
        "contexts": processed_contexts,
        "ground_truth": sub_df["ground_truth"].tolist(),
    }
    ragas_dataset = Dataset.from_dict(dataset_dict)
    
    try:
        result = evaluate(
            ragas_dataset,
            metrics=metrics,
            llm=llm,
            embeddings=embeddings,
            run_config=config
        )
        score_dict = result.to_pandas().mean(numeric_only=True).to_dict()
        score_dict["system_name"] = system_name
        system_scores.append(score_dict)
        
        # Sistem bittiği an ara karnesini yazdırıyoruz
        print(f"\n[✓] Başarılı! {system_name} için hesaplanan metrik skorları:")
        temp_df = pd.DataFrame([score_dict])
        cols_order = ["system_name"] + [c for c in temp_df.columns if c != "system_name"]
        print(temp_df[cols_order].to_string(index=False))
        print("-" * 60)
        
    except Exception as e:
        print(f"   [!] {system_name} değerlendirilirken hata oluştu: {str(e)}")

# 7. Sonuçları Derleme ve Kaydetme
if system_scores:
    df_final_ragas = pd.DataFrame(system_scores)
    cols = ["system_name"] + [c for c in df_final_ragas.columns if c != "system_name"]
    df_final_ragas = df_final_ragas[cols]

    print("\n" + "="*80)
    print("🎯 4'LÜ SİSTEM RAGAS DEĞERLENDİRME SONUÇ KARNESİ (FİNAL)")
    print("="*80)
    print(df_final_ragas.to_string(index=False))
    print("="*80)

    output_path = "/kaggle/working/ragas_dort_sistem_kesin_sonuc_karnesi.csv"
    df_final_ragas.to_csv(output_path, index=False, encoding="utf-8")
    print(f"\n[✓] RAGAS sonuçları başarıyla kaydedildi: {output_path}")
else:
    print("\n[!] Hiçbir sistemden skor alınamadı.")